In [ ]:
#read the retail sales data in from MySQL db and melt() to time series

import pandas as pd
#pd.set_option('display.max_rows', 3000)
pd.reset_option('display.max_rows')
#pd.set_option('display.max_columns', 357)
pd.reset_option('display.max_columns')

import yaml
import sqlalchemy
from sqlalchemy import create_engine, text

import pymysql #or mysql-connector-python
import matplotlib.pyplot as plt
from datetime import datetime

#retail sales database structures
DATABASE_NAME = "retail_sales_m8"
TABLE_NAME = "retail_sales_1992_2021"


#MySQL connection
yaml_file = r'C:\Users\kedre\OneDrive\Desktop\sample\db_retail_sales_SQLAlchemy.yaml'
def get_db_engine(config_file=yaml_file):
    """
    Loads database configuration from a YAML file and creates a SQLAlchemy engine.
    """
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)

    db_config = config['database']
    db_url = (
        f"mysql+pymysql://{db_config['user']}:{db_config['password']}@"
        f"{db_config['host']}:{db_config['port']}/{db_config['db']}"
    )
    engine = create_engine(db_url)
    return engine

if __name__ == "__main__": #make this code modular (this block only executes if this program name)
    engine = get_db_engine()
    
    try:
        with engine.connect() as connection:
            #print("Successfully connected to the MySQL database!")
            result = connection.execute(text(f"SELECT * FROM {TABLE_NAME}"))
            rows = result.fetchall()
            column_names = result.keys()
            retail_sales_df = pd.DataFrame(rows, columns=column_names)
            #print("Successful SELECT!")

    except Exception as e:
        print(f"Error connecting to the database: {e}")

#convert to time series: pivot the panel so date is filled out and switched to row
#create a list of the month columns to melt
month_columns = [f'_{i}' for i in range(1, 13)]

#print("Original DataFrame (first 5 rows):")
#print(retail_sales_df.head())

# Melt the DataFrame
retail_sales_df_melted = pd.melt(
    retail_sales_df,
    id_vars=['business_type', 'year'],
    value_vars=month_columns,
    var_name='Month_num',
    value_name='sales'
)

#print("\nMelted DataFrame (first 5 rows):")
#print(retail_sales_df_melted.head())

#convert the 'Month_num' column to an integer and remove the leading underscore
retail_sales_df_melted['Month_num'] = retail_sales_df_melted['Month_num'].str.lstrip('_').astype(int)

#create a new 'Date' column by combining Year and Month
retail_sales_df_melted['date'] = pd.to_datetime(retail_sales_df_melted['year'].astype(str) + '-' + retail_sales_df_melted['Month_num'].astype(str) + '-01')

#print("\nDataFrame with Date column (first 5 rows):")
#print(retail_sales_df_melted.head())

#drop year and Month_num cols
retail_sales_df_ts = retail_sales_df_melted.drop(columns=['year', 'Month_num'])

#sort
retail_sales_df_ts = retail_sales_df_ts.sort_values(by=['business_type', 'date']).reset_index(drop=True)

print("\nFinal DataFrame (first 5 rows):")
print(retail_sales_df_ts.head())

        



Final DataFrame (first 5 rows):
                       business_type  sales       date
0  All other gen. merchandise stores   2111 1992-01-01
1  All other gen. merchandise stores   2149 1992-02-01
2  All other gen. merchandise stores   2229 1992-03-01
3  All other gen. merchandise stores   2427 1992-04-01
4  All other gen. merchandise stores   2494 1992-05-01
